In [3]:
# https://platform.olimpiada-ai.ro/en/problems/220

import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.spatial.distance import mahalanobis

In [4]:
train = pl.read_csv("/kaggle/input/datasets/abukanabek/credit-card-fraud/train.csv")
test = pl.read_csv("/kaggle/input/datasets/abukanabek/credit-card-fraud/test.csv")
subm = pl.read_csv("/kaggle/input/datasets/abukanabek/credit-card-fraud/sample_output.csv")

In [5]:
train.shape, test.shape, subm.shape

((227845, 32), (56962, 31), (56964, 3))

In [6]:
train.head()

id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
1,143352.0,1.955041,-0.380783,-0.315013,0.330155,-0.509374,-0.086197,-0.627978,0.035994,1.05456,-0.030441,0.624996,1.691496,1.25579,-0.253266,-0.331695,0.307252,-0.930844,0.651666,0.167987,-0.12539,0.238197,0.968305,0.053208,-0.278602,-0.044999,-0.21678,0.045168,-0.047145,9.99,0
2,117173.0,-0.400975,-0.626943,1.555339,-2.017772,-0.107769,0.16831,0.017959,-0.401619,0.040378,0.611115,-1.94507,-0.726597,1.060888,-1.193347,0.631053,-0.160123,-1.630444,2.106866,-1.69278,-0.470372,-0.153485,0.421703,0.113442,-1.004095,-1.176695,0.361924,-0.370469,-0.144792,45.9,0
3,149565.0,0.072509,0.820566,-0.561351,-0.709897,1.080399,-0.359429,0.787858,0.117276,-0.131275,-0.638222,0.521931,-0.072768,-1.008237,-0.640249,-0.801946,0.678131,0.044374,0.521919,0.198772,0.012227,-0.314638,-0.872959,0.083391,0.148178,-0.431459,0.11969,0.206395,0.070288,11.99,0
4,93670.0,-0.535045,1.014587,1.750679,2.76939,0.500089,1.00227,0.847902,-0.081323,0.371579,0.560595,-0.855437,-4.179628,0.286872,1.271254,-1.011647,1.4586,-0.61326,0.814931,-2.147124,-0.253757,0.063525,0.443431,-0.072754,0.448192,-0.655203,-0.181038,-0.093013,-0.064931,117.44,0
5,82655.0,-4.026938,1.897371,-0.429786,-0.029571,-0.855751,-0.480406,-0.435632,1.31376,0.536044,1.221746,0.472626,1.595929,0.777603,0.187685,-1.060579,0.143332,0.007803,-0.055817,0.712695,-0.01232,-0.480691,-0.230369,0.250717,0.066399,0.470787,0.245335,0.286904,-0.322672,25.76,0


In [7]:
train['Class'].value_counts()

Class,count
i64,u32
1,394
0,227451


In [8]:
subtask1 = float((train.filter(pl.col('Class')==1)['Amount'] > train.filter(pl.col('Class')==0)['Amount'].mean()).sum())
subtask1

127.0

In [9]:
numeric_features = ['Amount'] + [f'V{i}' for i in range(1, 28+1)]

numerics = train.filter(pl.col('Class')==1).select(numeric_features).to_numpy()

mean_vector = numerics.mean(axis=0)
cov_matrix = np.cov(numerics.T)

eps = 1e-6
cov_matrix += eps * np.eye(cov_matrix.shape[0])
inv_cov_matrix = np.linalg.inv(cov_matrix)

mean_vector.shape, inv_cov_matrix.shape

((29,), (29, 29))

In [10]:
subtask2 = round(np.array([mahalanobis(mean_vector, numerics[i], inv_cov_matrix) for i in range(len(numerics))]).mean().item(), 2)
subtask2

5.1

In [11]:
from sklearn.model_selection import train_test_split
from catboost import Pool

features = ['Time'] + numeric_features
target_col = 'Class'

X, y = train.select(features).to_numpy(), train.select(target_col).to_numpy()
X_test = test.select(features).to_numpy()

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

train_pool = Pool(X_train, y_train)
valid_pool = Pool(X_valid, y_valid)

In [12]:
from catboost import CatBoostClassifier

params = {
    'iterations': 500,
    'loss_function': 'Logloss',
    'metric_period': 100,
    'eval_metric': 'F1',
    'random_state': 42,
    'max_depth': 2,
    'learning_rate': 0.1
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: 0.0000000	test: 0.0000000	best: 0.0000000 (0)	total: 100ms	remaining: 50s
100:	learn: 0.8438949	test: 0.8421053	best: 0.8421053 (100)	total: 2.2s	remaining: 8.69s
200:	learn: 0.8686244	test: 0.8533333	best: 0.8533333 (200)	total: 4.25s	remaining: 6.33s
300:	learn: 0.8854489	test: 0.8648649	best: 0.8648649 (300)	total: 6.3s	remaining: 4.16s
400:	learn: 0.8957055	test: 0.8648649	best: 0.8648649 (300)	total: 8.31s	remaining: 2.05s
499:	learn: 0.9054878	test: 0.8767123	best: 0.8767123 (499)	total: 10.4s	remaining: 0us

bestTest = 0.8767123288
bestIteration = 499



CatBoostClassifier(eval_metric='F1', iterations=500, learning_rate=0.1, loss_function='Logloss', max_depth=2, metric_period=100, random_state=42)

In [13]:
y_pred = model.predict(X_test)
y_pred, np.bincount(y_pred)

(array([1, 0, 0, ..., 0, 0, 0]), array([56883,    79]))

In [14]:
subm = subm.with_columns(answer=np.concatenate([np.array([subtask1, subtask2]), y_pred]))

In [15]:
subm.write_csv('submission.csv')
subm.head(10)

subtaskID,datapointID,answer
i64,i64,f64
1,1,127.0
2,1,5.1
3,227846,1.0
3,227847,0.0
3,227848,0.0
3,227849,0.0
3,227850,0.0
3,227851,0.0
3,227852,0.0
